In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.express as px

pio.renderers.default = 'notebook'
ROOT = Path.cwd() if (Path.cwd() / 'dashboard/app.py').is_file() else Path.cwd().parent
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT  # Для проверки можно указать отдельную временную папку.


# Единая подготовка данных
Читаем четыре исходных Excel, очищаем их и сохраняем два parquet. Отдельные скрипты сборки
и предварительно подготовленный CSV не требуются. Функции ниже используются в последующих
разделах; графики и пояснения остаются рядом с обработкой.

In [2]:
import re
import numpy as np
import pandas as pd

AGE_GROUPS = ['до 25', '25–34', '35–44', '45–54', '55–64', '65+']

# Статусы мобильности (по наличию педагога в данных следующего года)
STAY, MOVE, LEAVE = 0, 1, 2
MOVEMENT_LABELS = {STAY: 'остался', MOVE: 'перешёл в другую школу', LEAVE: 'ушёл из системы'}

MAX_STUDENTS_PER_TEACHER = 60

HIGHER_EDU = ('Высшее', 'Аспирантура')
PEDAGOGICAL = re.compile(r'педагог|учител|преподав', re.IGNORECASE)
SHORT_COURSE = re.compile(r'ппэ|организатор|подготовка|эксперт', re.IGNORECASE)

# Слова в "Направлении деятельности", которые не говорят о предмете
SUBJECT_STOPWORDS = {'языка', 'классов', 'культуры', 'основ', 'группы', 'дня', 'по', 'и', 'в'}
# Специализации, которые засчитываем как совпадение для целых групп предметов
SUBJECT_SYNONYMS = {
    'иностр': ['иностран', 'лингвист', 'филолог'],   # английского, немецкого, ... языка
    'русск': ['филолог'],
    'литер': ['филолог'],
    'начал': ['начальн'],
    'алгеб': ['матем'],
    'геоме': ['матем'],
    'общес': ['истор', 'социал', 'правов', 'эконом'],
    'изобр': ['художест', 'искусств', 'дизайн'],
    'техно': ['труд', 'технолог'],
    'безоп': ['безопасн'],
}
FOREIGN = ('английск', 'немецк', 'французск', 'испанск', 'китайск', 'итальянск', 'арабск')


# ---------------------------------------------------------------------------
# База: те же шаги, что в ноутбуке "Движуха в кадрах"
# ---------------------------------------------------------------------------
def has_information(series):
    return series.notna() & ~series.astype('string').str.strip().str.lower().isin(
        ['', '-', 'не указано', 'нет / не указано'])


def label_mobility(records):
    t = records[['id', 'year', 'school_num']].drop_duplicates().copy()
    assert not t.duplicated(['id', 'year']).any(), 'У педагога две школы в одном году'
    nxt = t.rename(columns={'school_num': 'next_school'}).copy()
    nxt['year'] -= 1
    t = t.merge(nxt, on=['id', 'year'], how='left', validate='one_to_one')
    known = t['year'] < t['year'].max()
    status = np.select([t['next_school'].isna(), t['next_school'] != t['school_num']],
                       [LEAVE, MOVE], default=STAY)
    t['mobility_status'] = pd.Series(np.where(known, status, np.nan), index=t.index).astype('Int8')
    t['movement_type'] = t['mobility_status'].map(MOVEMENT_LABELS)
    t['target'] = t['mobility_status'].eq(LEAVE).astype('Float64').astype(float)
    return t.drop(columns='next_school')


def build_base(df_clean, schools):
    fields = ['school_num', 'main_position_cat', 'age', 'experience_age_pedagogic_clean',
              'experience_age_all_clean', 'experience_age_other_clean', 'experience_valid', 'category_qual_cat',
              'category_due_clean', 'class_manager', 'sex', 'academic_degree_cat', 'target']
    conflicts = df_clean.groupby(['id', 'year'])[fields].nunique(dropna=False)
    assert not conflicts.gt(1).any().any(), 'Противоречивые кадровые поля внутри id, year'
    teachers = df_clean.drop_duplicates(['id', 'year']).copy()
    teachers['age_group'] = pd.cut(teachers['age'], bins=[0, 24, 34, 44, 54, 64, np.inf],
                                   labels=AGE_GROUPS, include_lowest=True)
    assert not schools.duplicated('school_num').any(), 'Повтор школы в справочнике'
    teachers = teachers.merge(schools[['school_num', 'district']], on='school_num',
                              how='left', validate='many_to_one')
    teachers = teachers[[
        'id', 'year', 'school_num', 'district', 'main_position_cat', 'age', 'age_group',
        'experience_age_pedagogic_clean', 'experience_age_all_clean', 'category_qual_cat',
        'category_due_clean', 'class_manager', 'speciality', 'sex', 'academic_degree_cat',
        'experience_age_other_clean', 'experience_valid',
    ]].reset_index(drop=True)
    return teachers.merge(label_mobility(df_clean), on=['id', 'year', 'school_num'],
                          how='left', validate='one_to_one')


# ---------------------------------------------------------------------------
# Нагрузка: предметы и параллели (из построчных записей "Педагогов")
# ---------------------------------------------------------------------------
def add_load(t: pd.DataFrame, df_clean: pd.DataFrame, schools: pd.DataFrame) -> pd.DataFrame:
    load = (df_clean.groupby(['id', 'year'])
            .agg(n_subjects=('subject', 'nunique'), n_parallels=('parallel', 'nunique'),
                 min_parallel=('parallel', 'min'), max_parallel=('parallel', 'max'))
            .reset_index())
    pairs = (df_clean[['id', 'year', 'subject', 'parallel']].drop_duplicates()
             .groupby(['id', 'year']).size().rename('n_subject_parallel').reset_index())
    t = t.merge(load, on=['id', 'year'], how='left').merge(pairs, on=['id', 'year'], how='left')

    # Учеников на педагога: контингент школы (один срез) / педагоги школы в этом году
    staff = t.groupby(['school_num', 'year'])['id'].nunique().rename('school_teachers').reset_index()
    contingent = schools[['school_num', 'students_num']].rename(columns={'students_num': 'school_students'})
    staff = staff.merge(contingent, on='school_num', how='left')
    ratio = staff['school_students'] / staff['school_teachers']
    # У части школ в данных лишь несколько педагогов (медиана 5) - штат выгружен не целиком.
    # В распределении разрыв: полные школы ≤ 60 учеников на педагога, неполные - от 100.
    staff['school_staff_complete'] = (ratio <= MAX_STUDENTS_PER_TEACHER).astype(int)
    staff['students_per_teacher'] = ratio.where(staff['school_staff_complete'] == 1)
    return t.merge(staff, on=['school_num', 'year'], how='left')


# ---------------------------------------------------------------------------
# Образование
# ---------------------------------------------------------------------------
def _subject_stems(speciality: str) -> list[str]:
    words = [w for w in str(speciality).lower().split() if w not in SUBJECT_STOPWORDS]
    stems = [w[:5] if len(w) >= 6 else w[:-1] for w in words if len(w) > 2]
    if any(w.startswith(FOREIGN) for w in words):
        stems.append('иностр')
    return stems + [s for st in stems for s in SUBJECT_SYNONYMS.get(st, [])]


def add_education(t: pd.DataFrame, edu: pd.DataFrame) -> pd.DataFrame:
    edu = edu.copy()
    edu['text'] = (edu['edu_org_name'].fillna('') + ' ' +
                   edu['specialization_edu'].fillna('') + ' ' + edu['qualification'].fillna('')).str.lower()
    edu['is_higher'] = edu['type_education'].str.startswith(HIGHER_EDU, na=False)
    edu['is_dpo'] = edu['type_education'].str.startswith('Дополнительное', na=False)
    base = edu[~edu['is_dpo']]

    g = base.groupby(['id', 'year'])
    per = pd.DataFrame({
        'edu_higher': g['is_higher'].any(),
        'edu_pedagogical': (base['is_higher'] & base['text'].str.contains(PEDAGOGICAL))
                           .groupby([base['id'], base['year']]).any(),
        'edu_text': g['text'].agg(' | '.join),
    }).reset_index()
    t = t.merge(per, on=['id', 'year'], how='left')

    # Совпадение специализации с предметом - эвристика по основам слов.
    # Считаем только для учителей: у руководителей "направление" - не предмет.
    def match(row):
        if (row['main_position_cat'] != 'учитель' or row['speciality'] == 'не указано'
                or pd.isna(row['edu_text'])):
            return np.nan
        stems = _subject_stems(row['speciality'])
        return float(any(s in row['edu_text'] for s in stems)) if stems else np.nan

    t['edu_speciality_match'] = t.apply(match, axis=1)
    t = t.drop(columns='edu_text')
    for c in ('edu_higher', 'edu_pedagogical'):
        t[c] = t[c].astype('boolean').fillna(False).astype(int)
    return t


# ---------------------------------------------------------------------------
# Повышение квалификации и переподготовка
# ---------------------------------------------------------------------------
def add_profdev(t: pd.DataFrame, edu: pd.DataFrame, pk: pd.DataFrame) -> pd.DataFrame:
    dpo = edu[edu['type_education'].str.startswith('Дополнительное', na=False)]
    # ДПО со специализацией/квалификацией = переподготовка (ПК их не даёт),
    # кроме курсов вроде "Организатор ППЭ", записанных в квалификацию
    short_course = dpo['qualification'].str.contains(SHORT_COURSE, na=False)
    is_pp = dpo['has_specialization'] | (dpo['has_qualification'] & ~short_course)
    pp_edu = dpo[is_pp][['id', 'year']]
    pp_pk = pk[pk['type_prog'] == 'ПП'][['id', 'year']]
    pp = pd.concat([pp_edu, pp_pk]).drop_duplicates()

    # Даты обучения: из ДПО в "Образовании" (история) и из листов ПК-ПП (текущий год)
    # Дата события и год появления записи различаются: будущую выгрузку не используем раньше срока.
    events = pd.concat([
        dpo.loc[dpo['date_known'] & ~dpo['Дата исправлена'].fillna(False), ['id', 'year', 'date_pass']].rename(
            columns={'year': 'source_year', 'date_pass': 'done'}),
        pk[['id', 'year']].assign(done=pk['year']).rename(columns={'year': 'source_year'}),
    ]).dropna().drop_duplicates()
    out = t[['id', 'year']].copy()
    ev = out.merge(events, on='id')
    ev = ev[(ev['source_year'] <= ev['year']) & (ev['done'] <= ev['year'])
            & (ev['done'] >= ev['year'] - 2)]
    recent = ev[['id', 'year']].drop_duplicates().assign(pk_recent=1)

    # Часы есть только в ПК-ПП (листы 2022–2024): сумма за окно [Y-2, Y]
    hours = out.merge(pk[['id', 'year', 'hours_tot']].rename(columns={'year': 'pk_year'}), on='id')
    hours = hours[(hours['pk_year'] <= hours['year']) & (hours['pk_year'] >= hours['year'] - 2)]
    hours = hours.groupby(['id', 'year'])['hours_tot'].sum().rename('pk_hours_3y').reset_index()
    pk_year = pk.groupby(['id', 'year']).size().rename('pk_n').reset_index()

    t = (t.merge(recent, on=['id', 'year'], how='left')
          .merge(hours, on=['id', 'year'], how='left')
          .merge(pk_year, on=['id', 'year'], how='left')
          .merge(pp.assign(retraining=1), on=['id', 'year'], how='left'))
    t['pk_recent'] = t['pk_recent'].fillna(0).astype(int)
    t['pk_hours_3y'] = t['pk_hours_3y'].fillna(0)
    t['pk_this_year'] = t['pk_n'].notna().astype(int)
    t['retraining'] = t['retraining'].fillna(0).astype(int)
    return t.drop(columns='pk_n')



# Предобработка

## Учителя

In [3]:
df_pedagog1 = pd.read_excel(DATA_DIR / 'Педагоги.xlsx', sheet_name='2022')
df_pedagog1['year']=2022

df_pedagog2 = pd.read_excel(DATA_DIR / 'Педагоги.xlsx', sheet_name='2023')
df_pedagog2['year']=2023

df_pedagog3 = pd.read_excel(DATA_DIR / 'Педагоги.xlsx', sheet_name='2024')
df_pedagog3['year']=2024

df_pedagog_all = pd.concat([df_pedagog1, df_pedagog2, df_pedagog3], axis=0)

In [4]:
df_pedagog_all.head(5)

,id,ID образовательной организации,Основная должность,Направление деятельности,Пол,Возраст,Ученая степень,Квалификационная категория (педагогического работника),Дата присвоения квалификационной категории,Срок действия квалификационной категории,Общий стаж работы,Стаж педагогической работы,Предмет,Параллель,Имеет классное руководство,year
0,f583c34a49,478,Педагог дополнительного образования,NaN,Женский,35,NaN,NaN,NaN,NaN,12,1,Иностранный язык (английский),2,Нет,2022
1,f583c34a49,478,Педагог дополнительного образования,NaN,Женский,35,NaN,NaN,NaN,NaN,12,1,Иностранный язык (английский),6,Нет,2022
2,f583c34a49,478,Педагог дополнительного образования,NaN,Женский,35,NaN,NaN,NaN,NaN,12,1,Практикум,6,Нет,2022
3,f583c34a49,478,Педагог дополнительного образования,NaN,Женский,35,NaN,NaN,NaN,NaN,12,1,Иностранный язык (английский),9,Нет,2022
4,08ff701e26,523,Учитель,NaN,Женский,54,NaN,NaN,NaN,NaN,29,24,Физика,7,Нет,2022


In [5]:
df_pedagog_all = df_pedagog_all.rename(columns={'ID образовательной организации':'school_num',
                                                'Основная должность':'main_position',
                                                'Направление деятельности':'speciality',
                                                'Пол': 'sex',
                                                'Возраст':'age',
                                                'Ученая степень':'academic_degree',
                                                'Квалификационная категория (педагогического работника)':'category_qual',
                                                'Дата присвоения квалификационной категории':'category_date',
                                                'Срок действия квалификационной категории':'category_due',
                                                'Общий стаж работы':'experience_age_all',
                                                'Стаж педагогической работы':'experience_age_pedagogic',
                                                'Предмет':'subject',
                                                'Параллель':'parallel',
                                                'Имеет классное руководство':'class_manager'})

### main_position - Основная должность

In [6]:
df_pedagog_all.main_position.value_counts()

main_position
Учитель                                                        399283
Педагог дополнительного образования                              5279
Педагог-организатор                                              2731
Учитель-дефектолог                                               2596
Воспитатель                                                      2239
                                                                ...  
заместитель директора                                               1
Заместитель директора по качеству образовательного процесса         1
Наладчик станков и манипуляторов с программным управлением          1
Технический редактор                                                1
Руководитель проектов                                               1
Name: count, Length: 133, dtype: int64

В списке присутствует большое количество разных должностей, часть из которых могут не иметь отношения к преподавания  
Посмотрим на наличие слова "учитель" в списке должностей. Там присутствуют разные варианты написания:

In [7]:
matching = set([item for item in df_pedagog_all.main_position.values
            if any(sub in str(item).lower() for sub in ['учитель'])])
print(',\n'.join(list(matching)))

Учитель начальных классов,
Учитель информатики,
Учитель  (не этал.),
учитель,
Учитель-логопед,
Учитель русского языка,
Учитель музыки,
Учитель истории,
Учитель-дефектолог,
Учитель русского языка и литературы,
Учитель - дефектолог,
Учитель географии,
УЧИТЕЛЬ,
Учитель английского языка,
учитель истории и обществознания,
Учитель ИЗО,
Учитель технологии,
Учитель физической культуры,
Учитель математики,
Учитель китайского языка,
Учитель,
Учитель биологии,
Учитель физики,
Учитель иностранного языка,
Учитель, 4 квалификационный уровень,
Учитель химии


Попробуем определить тех, кто работает по основному месту, а кто совместитель

In [8]:
keywords = ['педагог', 'учитель', 'методист', 'преподават', 'воспитатель', 'тьютор']
matching_pedagog = set([item for item in df_pedagog_all.main_position.values
            if any(sub in str(item).lower() for sub in keywords)])
matching_pedagog = set([item for item in matching_pedagog if 'заместитель' not in str(item).lower()]) # иногда встречаются созвучные должности, например, Заместитель директора по воспитательной работе и социализации - уберем

In [9]:
matching_pedagog

{'Воспитатель',
 'Воспитатель (ГПиУ)',
 'Воспитатель (кадетского класса)',
 'Воспитатель кадетских классов',
 'Методист',
 'Олигофренопедагог',
 'Педагог дополнительного  образования',
 'Педагог дополнительного образования',
 'Педагог допонительного образования',
 'Педагог организатор',
 'Педагог организатор безопасности жизнедеятельности',
 'Педагог- организатор',
 'Педагог-библиотекарь',
 'Педагог-организатор',
 'Педагог-организатор ОБЖ',
 'Педагог-психолоГ',
 'Педагог-психолог',
 'Преподаватель',
 'Преподаватель - организатор основ безопасности жизнедеятельности',
 'Преподаватель-организатор ОБЖ',
 'Преподаватель-организатор основ безопасности жизнедеятельности',
 'Социальный педагог',
 'Старший воспитатель',
 'Старший методист',
 'Старший методист по руководству воспитательной работой',
 'Старший педагог дополнительного образования',
 'Старший педагог-психолог',
 'Тренер-преподаватель',
 'Тьютор',
 'УЧИТЕЛЬ',
 'Учитель',
 'Учитель  (не этал.)',
 'Учитель - дефектолог',
 'Учитель ИЗ

Аналогичным образом можно определить тех, кто работает на руководящей должности

In [10]:
keywords_manager = ['заместитель', 'ведущий', 'заведующий', 'руководитель', 'начальник', 'главный', 'директор', 'советник']
matching_manager = set([item for item in df_pedagog_all.main_position.values
            if any(sub in str(item).lower() for sub in keywords_manager)])

In [11]:
matching_other = set(df_pedagog_all.main_position.unique()).difference(matching_pedagog.union(matching_manager)) # остальные совместители
matching_other

{'Администратор',
 'Администратор баз данных',
 'Ассистент',
 'Ассистент по оказанию технической помощи',
 'Библиотекарь',
 'Вожатый',
 'Делопроизводитель',
 'Диспетчер',
 'Диспетчер образовательного учреждения',
 'Документовед',
 'Звукооператор',
 'Инженер',
 'Инженер по безопасности',
 'Инженер-электроник',
 'Инструктор по труду',
 'Инструктор по физической культуре',
 'Контрактный управляющий',
 'Концертмейстер',
 'Лаборант',
 'Мастер производственного обучения',
 'Наладчик станков и манипуляторов с программным управлением',
 'Помощник воспитателя',
 'Рабочий по комплексному обслуживанию и ремонту зданий',
 'Режиссер',
 'Секретарь учебной части',
 'Системный администратор',
 'Системный администратор информационно-коммуникационных систем',
 'Специалист',
 'Специалист по гражданской обороне и защите в чрезвычайных ситуациях',
 'Специалист по интернет-маркетингу',
 'Специалист по кадрам',
 'Специалист по охране труда',
 'Специалист по связям с общественностью',
 'Специалист по сетевому

Создадим новый столбец с категориями основных должностей: учитель, руководитель, другой

In [12]:
matching_dict = {}

for val in matching_pedagog:
    matching_dict[val] = 'учитель'
for val in matching_manager:
    matching_dict[val] = 'руководитель'
for val in matching_other:
    matching_dict[val] = 'другой'

In [13]:
df_pedagog_all['main_position_cat'] = df_pedagog_all['main_position'].map(matching_dict).fillna('не_найдено')

In [14]:
print(f"\nРаспределение:")
print(df_pedagog_all['main_position_cat'].value_counts())


Распределение:
main_position_cat
учитель         423935
руководитель      2272
другой            1933
Name: count, dtype: int64


### speciality - Направление деятельности

In [15]:
df_pedagog_all.speciality.value_counts()

speciality
начальных классов                                                   68639
русского языка                                                      37555
английского языка                                                   35651
математики                                                          33280
истории                                                             24184
информатики                                                         18221
физической культуры                                                 16309
биологии                                                            11475
физики                                                              10308
географии                                                            8202
химии                                                                7332
музыки                                                               3687
технологии                                                           3496
изобразительного искусства 

### sex - Пол

In [16]:
df_pedagog_all.sex.unique()

<ArrowStringArray>
['Женский', 'Мужской']
Length: 2, dtype: str

Приводим в бинарный вид

In [17]:
df_pedagog_all['sex'] = df_pedagog_all['sex'].map({'Мужской': 1, 'Женский': 0})

### age - Возраст

In [18]:
df_pedagog_all.age.unique()

array([35, 54, 52, 41, 58, 55, 46, 28, 23, 56, 32, 61, 29, 22, 39, 30, 51,
       45, 47, 26, 40, 50, 25, 34, 49, 27, 62, 63, 33, 38, 53, 48, 60, 65,
       76, 31, 59, 73, 42, 68, 44, 57, 43, 37, 24, 66, 67, 74, 77, 36, 64,
       21, 20, 69, 18, 19, 70, 72, 75, 80, 71, 78, 81, 88, 17, 79, 82, 86,
       83, 16])

### academic_degree - Ученая степень

In [19]:
df_pedagog_all.academic_degree.unique()

<ArrowStringArray>
[                                                         nan,
                                 'Кандидат исторических наук',
                                  'Кандидат юридических наук',
                               'Кандидат филологических наук',
                                  'Кандидат философских наук',
                                              'Кандидат наук',
                               'Кандидат педагогических наук',
                              'Кандидат психологических наук',
                                 'Кандидат политических наук',
         'Доктор технических наук, Кандидат технических наук',
                                   'Кандидат химических наук',
                                'Кандидат экономических наук',
                                  'Кандидат технических наук',
                                'Кандидат биологических наук',
                        'Кандидат физико-математических наук',
                                  'К

In [20]:
keywords = {'кандидат': 'кандидат', 'доктор': 'доктор'}

def find_academic_degree(text):
    if pd.isna(text):
        return 'не_найдено'

    text = str(text).lower()
    for word, code in keywords.items():
        if word in text:
            return code
    return 'не_найдено'

df_pedagog_all['academic_degree_cat'] = df_pedagog_all['academic_degree'].apply(find_academic_degree)

In [21]:
df_pedagog_all.academic_degree_cat.value_counts()

academic_degree_cat
не_найдено    418850
кандидат        9223
доктор            67
Name: count, dtype: int64

### category_qual - Квалификационная категория

In [22]:
df_pedagog_all.category_qual.value_counts()

category_qual
Высшая категория                      221551
Первая категория                      114272
-                                        956
Вторая категория                         785
Высшая категория, Первая категория       673
Вторая категория, Первая категория        13
Name: count, dtype: int64

In [23]:
keywords = {'высшая': 'высшая', 'первая': 'первая', 'вторая': 'вторая'}

def find_category_qual(text):
    if pd.isna(text):
        return 'не_найдено'

    text = str(text).lower()
    for word, code in keywords.items():
        if word in text:
            return code
    return 'не_найдено'

# Применяем функцию
df_pedagog_all['category_qual_cat'] = df_pedagog_all['category_qual'].apply(find_category_qual)

In [24]:
print(f"\nРаспределение:")
print(df_pedagog_all['category_qual_cat'].value_counts())


Распределение:
category_qual_cat
высшая        222224
первая        114285
не_найдено     90846
вторая           785
Name: count, dtype: int64


### category_date и category_due - действие квалификационной категории

In [25]:
df_pedagog_all['category_date'] = df_pedagog_all['category_date'].astype('Int64')
df_pedagog_all['category_due'] = df_pedagog_all['category_due'].astype('Int64')

In [26]:
# Годы сравниваем как числа; равный год сохраняет прежнее правило «не активно».
df_pedagog_all['category_active'] = np.select(
    [df_pedagog_all['category_due'].isna(),
     df_pedagog_all['category_due'].gt(df_pedagog_all['year']).fillna(False)],
    ['не было', 'активно'], default='не активно',
)


In [27]:
print(f"\nДоля с активной категорией: {(df_pedagog_all['category_active']=='активно').mean():.4f}")
df_pedagog_all.category_active.value_counts()


Доля с активной категорией: 0.5841


category_active
активно       250074
не было        98295
не активно     79771
Name: count, dtype: int64

### experience_age_all и experience_age_pedagogic - стаж работы

In [28]:
df_pedagog_all.experience_age_all.unique()

array([12, 29, 34, 19, 33, 37, 27,  5,  3, 36,  9, 32, 43,  6,  1, 16,  7,
       20, 24, 11,  4, 18, 30, 25,  2, 15, 31, 46, 40, 39, 23, 17, 38,  8,
       41, 35, 53, 50, 13, 10, 21, 28,  0, 22, 44, 26, 47, 54, 42, 48, 14,
       55, 45, 49, 52, 58, 51, 56, 57, 59, 63, 69, 60, 67, 70, 61, 64, 71])

In [29]:
df_pedagog_all.experience_age_pedagogic.unique()

array([ 1, 24, 30, 19, 21, 27,  5,  3,  7, 32, 35,  6, 29, 16, 15, 18,  9,
       11,  2,  0, 36, 10,  4, 28, 26, 37, 38,  8, 31, 23, 17, 34, 14, 53,
       13, 22, 25, 12, 40, 33, 43, 44, 20, 46, 39, 47, 41, 42, 50, 48, 51,
       49, 55, 45, 58, 56, 52, 64, 63, 54, 57, 67, 59])

Создаём производный признак: `experience_age_other` — разница между общим
стажем и педагогическим. Это показывает, сколько лет учитель работал
вне системы образования.

**Гипотеза:** учителя с большим опытом работы вне школы могут легче
вернуться в другую сферу при неудовлетворённости условиями.

In [30]:
df_pedagog_all['experience_age_other'] = df_pedagog_all.experience_age_all - df_pedagog_all.experience_age_pedagogic

### subject - Предмет

In [31]:
df_pedagog_all.subject.unique().size

np.int64(3459)

In [32]:
df_pedagog_all.subject.value_counts().iloc[:50]

subject
Русский язык                                           32451
Английский язык                                        27172
Математика                                             22839
Физическая культура                                    20572
Технология                                             18197
Литература                                             17971
Информатика                                            13970
Литературное чтение                                    13593
Окружающий мир                                         13423
Изобразительное искусство                              12377
Геометрия                                              11391
Обществознание                                         11218
Иностранный язык (английский)                          10691
Биология                                               10673
География                                               9311
Алгебра                                                 9045
Физика          

In [33]:
df_pedagog_all['n_unique_subjects'] = (
    df_pedagog_all.groupby(['year', 'id'])['subject'].transform('nunique')
)

**Важно:** Таким образом мы удобно выполняем группировку и сокращаем количество строк, но взамен происходит потеря значимой информации о самих проводимых предметах.

### parallel - Параллель

In [34]:
df_pedagog_all['min_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('min')
)

In [35]:
df_pedagog_all['max_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('max')
)

In [36]:
df_pedagog_all['n_unique_parallel'] = (
    df_pedagog_all.groupby(['year', 'id'])['parallel'].transform('nunique')
)

In [37]:
df_pedagog_all.iloc[:, 10:]

,experience_age_all,experience_age_pedagogic,subject,parallel,class_manager,year,main_position_cat,academic_degree_cat,category_qual_cat,category_active,experience_age_other,n_unique_subjects,min_parallel,max_parallel,n_unique_parallel
0,12,1,Иностранный язык (английский),2,Нет,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3
1,12,1,Иностранный язык (английский),6,Нет,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3
2,12,1,Практикум,6,Нет,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3
3,12,1,Иностранный язык (английский),9,Нет,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3
4,29,24,Физика,7,Нет,2022,учитель,не_найдено,не_найдено,не было,5,2,7,11,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153744,4,4,Музыка,2,Нет,2024,учитель,не_найдено,не_найдено,не было,0,1,2,6,5
153745,4,4,Музыка,3,Нет,2024,учитель,не_найдено,не_найдено,не было,0,1,2,6,5
153746,4,4,Музыка,4,Нет,2024,учитель,не_найдено,не_найдено,не было,0,1,2,6,5
153747,4,4,Музыка,5,Нет,2024,учитель,не_найдено,не_найдено,не было,0,1,2,6,5


### class_manager - Классный руководитель

In [38]:
df_pedagog_all.class_manager.unique()

<ArrowStringArray>
['Нет', 'Да']
Length: 2, dtype: str

In [39]:
df_pedagog_all['class_manager'] = df_pedagog_all['class_manager'].map({'Да': 1, 'Нет': 0})

Полный список итоговых столбцов для DataFrame с информацией об учителях:

In [40]:
df_pedagog_all.columns

Index(['id', 'school_num', 'main_position', 'speciality', 'sex', 'age',
       'academic_degree', 'category_qual', 'category_date', 'category_due',
       'experience_age_all', 'experience_age_pedagogic', 'subject', 'parallel',
       'class_manager', 'year', 'main_position_cat', 'academic_degree_cat',
       'category_qual_cat', 'category_active', 'experience_age_other',
       'n_unique_subjects', 'min_parallel', 'max_parallel',
       'n_unique_parallel'],
      dtype='str')

## Образование

In [41]:
df_edu1 = pd.read_excel(DATA_DIR / 'Образование.xlsx', sheet_name='2022')
df_edu1['year']=2022

df_edu2 = pd.read_excel(DATA_DIR / 'Образование.xlsx', sheet_name='2023')
df_edu2['year']=2023

df_edu3 = pd.read_excel(DATA_DIR / 'Образование.xlsx', sheet_name='2024')
df_edu3['year']=2024

df_edu_all = pd.concat([df_edu1, df_edu2, df_edu3], axis=0)

In [42]:
df_edu_all.head(5)

,id,Тип образования,Наименование учебного заведения,Дата окончания,Специализация,Квалификация,year
0,f583c34a49,"Высшее образование - специалитет, магистратура","ФГОУ ВПО ""Государственный технологический унив...",2006.0,Теория и методика преподавания иностранных язы...,"Лингвист, преподаватель",2022
1,08ff701e26,Высшее образование,Бийский государственный педагогический институт,1989.0,Физика,учитель физики и математики,2022
2,b813df019b,Высшее образование,МОПИ им. Н. К. Крупской,1993.0,Математика,Учитель математики,2022
3,b813df019b,Дополнительное профессиональное образование,-,2016.0,Педагогика,NaN,2022
4,b813df019b,Дополнительное профессиональное образование,-,2016.0,Педагогика,NaN,2022


In [43]:
df_edu_all = df_edu_all.rename(columns={
                               'Тип образования': 'type_education',
                               'Наименование учебного заведения': 'edu_org_name',
                               'Дата окончания': 'date_pass',
                               'Специализация': 'specialization_edu',
                               'Квалификация': 'qualification',
                               })

In [44]:
df_edu_all['date_pass'] = df_edu_all['date_pass'].astype('Int64')

### type_education - Тип образования

In [45]:
df_edu_all.type_education.value_counts()

type_education
Дополнительное профессиональное образование                   235012
Высшее образование                                             44894
Высшее образование - специалитет, магистратура                  8181
Среднее профессиональное образование                            8028
Высшее образование - бакалавриат                                5876
Среднее (полное) общее образование                               343
Аспирантура                                                      108
Среднее общее образование                                         98
Основное общее образование                                        66
Неполное высшее образование                                       66
Высшее образование - подготовка кадров высшей квалификации        47
Name: count, dtype: int64

In [46]:
def change_type_education(x):
    if x['type_education'] in ['Высшее образование - подготовка кадров высшей квалификации', 'Аспирантура']:
        return 'Аспирантура'
    elif x['type_education'] in ['Высшее образование', 'Высшее образование - специалитет, магистратура', 'Высшее образование - бакалавриат']:
        return 'Высшее образование'
    elif x['type_education'] in ['Среднее профессиональное образование']:
        return 'СПО'
    elif x['type_education'] in ['Неполное высшее образование', 'Среднее (полное) общее образование', 'Среднее общее образование', 'Основное общее образование']:
        return 'Школа'
    else:
        return 'Нет информации'

df_edu_all['high_type_education'] = df_edu_all.apply(lambda x: change_type_education(x), axis=1)

In [47]:
priority_type_education = ["Аспирантура", "Высшее образование", "СПО", 'Школа', 'Нет информации']

def get_high_type_education(group):
    for cat in priority_type_education:
        if cat in group.values:
            return cat
    return group.iloc[0]

df_edu_all['high_type_education'] = df_edu_all.groupby(['id', 'year'])['high_type_education'].transform(get_high_type_education)

### edu_org_name - Наименование учебного заведения

In [48]:
city = {
    'Москов': 'Москва',
    'Санкт': 'Спб',
    'Ленинград': 'Спб'
}

In [49]:
conditions = [df_edu_all['edu_org_name'].str.contains(keyword, na=False) for keyword in city.keys()]
choices = list(city.values())

df_edu_all['city'] = np.select(conditions, choices, default='Другой')

In [50]:
df_edu_all.city.value_counts()

city
Другой    271691
Москва     30618
Спб          410
Name: count, dtype: int64

## ПК и ПП

In [51]:
df_pkpp1 = pd.read_excel(DATA_DIR / 'ПК-ПП.xlsx', sheet_name='2022')
df_pkpp1['year']=2022

df_pkpp2 = pd.read_excel(DATA_DIR / 'ПК-ПП.xlsx', sheet_name='2023')
df_pkpp2['year']=2023

df_pkpp3 = pd.read_excel(DATA_DIR / 'ПК-ПП.xlsx', sheet_name='2024')
df_pkpp3['year']=2024

df_pkpp_all = pd.concat([df_pkpp1, df_pkpp2, df_pkpp3], axis=0)

In [52]:
df_pkpp_all

,id,Наименование программы,Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК),Количество часов,Дата получения документа/освоения программы,year
0,18ac5986cd,Методические подходы в подготовке к государств...,ПК,16,2022,2022
1,18ac5986cd,Организация образовательного процесса в услови...,ПК,22,2022,2022
2,18ac5986cd,Подготовка организаторов пунктов проведения эк...,ПК,16,2022,2022
3,9b13af5810,Подготовка организаторов пунктов проведения эк...,ПК,16,2022,2022
4,1be672d40b,Инструменты объективного оценивания уровня дос...,ПК,18,2022,2022
...,...,...,...,...,...,...
15730,a809c058df,Подготовка организаторов пунктов проведения эк...,ПК,18,2024,2024
15731,d45cfdfdfa,Подготовка организаторов пунктов проведения эк...,ПК,18,2024,2024
15732,d45cfdfdfa,Подготовка экспертов для работы в предметной к...,ПК,42,2024,2024
15733,0429862c8a,Подготовка руководителей пунктов проведения эк...,ПК,24,2024,2024


In [53]:
df_pkpp_all = df_pkpp_all.rename(columns={
                                'Наименование программы':'name_prog',
                                'Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК)':'type_prog',
                                'Количество часов':'hours_tot',
                                'Дата получения документа/освоения программы':'date_done',
                               })

In [54]:
keywords = ['преподавания', 'курса']
pattern = '|'.join(keywords)
df_pkpp_all['is_pedagog'] = df_pkpp_all['name_prog'].str.contains(pattern, na=False, case=False).astype(int)

In [55]:
df_pkpp_all['is_pedagog'].value_counts()

is_pedagog
0    64951
1      782
Name: count, dtype: int64

In [56]:
with pd.option_context('display.max_colwidth', None):
    print(df_pkpp_all[df_pkpp_all.is_pedagog==1]['name_prog'].head(10))

87                                                                                           Основы преподавания русского языка как иностранного
122                               Содержание и методика преподавания курса «Основы православной культуры» на уровне основного общего образования
146     Информационно-коммуникативные особенности медиадискурса английского языка при обучении школьников в рамках курса  «Медиа и журналистика»
625        Основы теории коммуникации в преподавании курса «Реклама и связи с общественностью» в рамках проекта «Медиакласс в московской школе» 
692        Основы теории коммуникации в преподавании курса «Реклама и связи с общественностью» в рамках проекта «Медиакласс в московской школе» 
783                                 Особенности обучения школьников в рамках элективного курса «Риторика» на уровне среднего общего образования 
821                              Содержание и методика преподавания курса «Основы православной культуры» на уровне начального обще

## Школы

In [57]:
df_school = pd.read_excel(DATA_DIR / 'Школы.xlsx')

In [58]:
df_school

,Школа,Округ,Кол-во зданий,Общая численность контингента
0,325,Инновационный округ,6,1294
1,397,Инновационный округ,3,842
2,192,Северный округ,13,4505
3,385,Западный округ,12,3617
4,269,Восточный округ,15,4262
...,...,...,...,...
261,522,Инновационный округ,12,778
262,32,Южный округ,25,2916
263,13,Южный округ,8,2399
264,219,Западный округ,23,5261


In [59]:
df_school = df_school.rename(columns={
                                'Школа':'school_num',
                                'Округ':'district',
                                'Кол-во зданий':'buildings_num',
                                'Общая численность контингента':'students_num',
                               })

Оставляем в df_pedagog_all только те строки, у которых school_num встречается в df_school. Это фильтрация по списку допустимых school_num

In [60]:
df_pedagog_all = df_pedagog_all[df_pedagog_all['school_num'].isin(df_school['school_num'])]

## Целевая переменная
### Шаг 1: Подготовка данных для поиска будущего состояния
Сначала нам нужно получить уникальные комбинации учитель - год - школа, чтобы избежать дублирования строк при объединении.
Затем мы создадим копию этих данных и "сдвинем" год на единицу назад.

In [61]:
# Единая разметка для подробной и кадровой таблиц.
target_df = label_mobility(df_pedagog_all)[['id', 'year', 'school_num', 'target']]
display(target_df.head())

,id,year,school_num,target
0,f583c34a49,2022,478,0.0
1,08ff701e26,2022,523,1.0
2,b813df019b,2022,42,0.0
3,b484e1de85,2022,66,0.0
4,60f45bfe9e,2022,19,0.0


### Разметка исходов
Используем одну функцию `label_mobility`: она сопоставляет именно соседние годовые срезы.
Отсутствие в следующем срезе имеет target=1, переход между школами — target=0.
Для последнего года target и статусы неизвестны. Граница системы остаётся прежней:
педагоги школ из предоставленного справочника. Это отсутствие в выборке системы,
а не документально подтверждённая причина увольнения.

In [62]:
display(target_df.groupby('year')['target'].agg(['size', 'count', 'sum']))

,size,count,sum
year,,,
2022,17396,17396,2596.0
2023,17394,17394,2299.0
2024,17933,0,0.0


### Шаг 3: Интеграция таргета в основной датасет.

In [63]:
if 'target' in df_pedagog_all.columns:
    df_pedagog_all = df_pedagog_all.drop(columns=['target'])

df_pedagog_all = df_pedagog_all.merge(target_df, on=['id', 'year', 'school_num'], how='left')

print("Распределение целевой переменной 'target':")
display(df_pedagog_all['target'].value_counts(dropna=False))

Распределение целевой переменной 'target':


target
0.0    234488
NaN    149690
1.0     32971
Name: count, dtype: int64

# Очистка

In [64]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

df = df_pedagog_all

print(f'Размер: {df.shape[0]:,} строк × {df.shape[1]} столбцов')
print(f'Школ: {df["school_num"].nunique():,}')
print(f'Уникальных пар педагог–год: {df[["id", "year"]].drop_duplicates().shape[0]:,}')

Размер: 417,149 строк × 26 столбцов
Школ: 266
Уникальных пар педагог–год: 52,723


In [65]:
df.head(5)

,id,school_num,main_position,speciality,sex,age,academic_degree,category_qual,category_date,category_due,experience_age_all,experience_age_pedagogic,subject,parallel,class_manager,year,main_position_cat,academic_degree_cat,category_qual_cat,category_active,experience_age_other,n_unique_subjects,min_parallel,max_parallel,n_unique_parallel,target
0,f583c34a49,478,Педагог дополнительного образования,NaN,0,35,NaN,NaN,<NA>,<NA>,12,1,Иностранный язык (английский),2,0,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3,0.00
1,f583c34a49,478,Педагог дополнительного образования,NaN,0,35,NaN,NaN,<NA>,<NA>,12,1,Иностранный язык (английский),6,0,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3,0.00
2,f583c34a49,478,Педагог дополнительного образования,NaN,0,35,NaN,NaN,<NA>,<NA>,12,1,Практикум,6,0,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3,0.00
3,f583c34a49,478,Педагог дополнительного образования,NaN,0,35,NaN,NaN,<NA>,<NA>,12,1,Иностранный язык (английский),9,0,2022,учитель,не_найдено,не_найдено,не было,11,2,2,9,3,0.00
4,08ff701e26,523,Учитель,NaN,0,54,NaN,NaN,<NA>,<NA>,29,24,Физика,7,0,2022,учитель,не_найдено,не_найдено,не было,5,2,7,11,5,1.00



Один педагог может встречаться несколько раз в одном году, потому что в исходной витрине отдельными строками представлены предметы и параллели. Для анализа предметной нагрузки это полезно, но для кадровых показателей (число педагогов, возраст, стаж, квалификационная категория) такие строки нельзя считать как разных людей.

Поэтому далее:
- для проверки качества исходных полей используем полный `df`;
- для показателей СППР создаём отдельную витрину `teachers` - **одна строка на `id + year`**.

In [66]:
rows_per_teacher = df.groupby(['id', 'year']).size()
print('Среднее число строк на педагога в год:', round(rows_per_teacher.mean(), 2))
print('Максимальное число строк на педагога в год:', rows_per_teacher.max())
print('Полностью одинаковых строк:', df.duplicated().sum())

Среднее число строк на педагога в год: 7.91
Максимальное число строк на педагога в год: 58


Полностью одинаковых строк: 53382


## Проверка и обработка пропусков

Сначала считаем количество и долю пропусков по всем столбцам. Отдельно важно отличать:

- **технический пропуск**, который мешает анализу и должен быть заполнен;
- **структурный пропуск**, у которого есть содержательный смысл. Например, дата квалификационной категории отсутствует у педагога без категории. Подставлять туда средний год было бы неверно.

In [67]:
missing = pd.DataFrame({
    'Пропусков': df.isna().sum(),
    'Доля, %': (df.isna().mean() * 100).round(2)
})
missing = missing[missing['Пропусков'] > 0].sort_values('Доля, %', ascending=False)
missing

,Пропусков,"Доля, %"
academic_degree,407949,97.79
target,149690,35.88
speciality,127747,30.62
category_due,93977,22.53
category_date,86456,20.73
category_qual,86329,20.70


### Обработка пропусков

Пропуски обрабатывались с учётом смысла признаков. В speciality используется значение не указано, чтобы не подменять отсутствующую информацию наиболее частой специальностью. В academic_degree пропуски заменяются на нет / не указано, а в category_qual - на нет категории, поскольку эти значения корректно отражают отсутствие соответствующего признака.

Пропуски в category_date и category_due не заполняются, так как даты имеют смысл только при наличии категории. Поле target также не заполняется его пропуски естественны.

Таким образом, заполняются только безопасные категориальные пропуски, а структурные пропуски сохраняются.

In [68]:
df_clean = df.copy()

# Текстовые пропуски заполняем явными категориями.
df_clean['speciality'] = df_clean['speciality'].fillna('не указано')
df_clean['academic_degree'] = df_clean['academic_degree'].fillna('нет / не указано')
df_clean['category_qual'] = df_clean['category_qual'].fillna('нет категории')


In [69]:
print('Осталось пропусков в speciality:', df_clean['speciality'].isna().sum())
print('Осталось пропусков в academic_degree:', df_clean['academic_degree'].isna().sum())
print('Осталось пропусков в category_qual:', df_clean['category_qual'].isna().sum())

Осталось пропусков в speciality: 0
Осталось пропусков в academic_degree: 0
Осталось пропусков в category_qual: 0


## 3. Проверка и обработка выбросов

Для первичного поиска применяем правило межквартильного размаха (IQR):

- нижняя граница: `Q1 - 1.5 × IQR`;
- верхняя граница: `Q3 + 1.5 × IQR`.

Но перед удалением значений нужно понять смысл отклонения, так как это может не быть выбросом.

In [70]:
def iqr_info(series):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    mask = (series < low) | (series > high)
    return low, high, int(mask.sum())

# Для кадровых показателей берём одну строку на педагог–год,
# чтобы повторения предметов не завышали число выбросов.
teacher_level_check = df_clean.drop_duplicates(['id', 'year']).copy()

numeric_checks = [
    'age',
    'experience_age_all',
    'experience_age_pedagogic',
    'experience_age_other',
    'n_unique_subjects',
    'category_date',
    'category_due',
]

outlier_rows = []
for col in numeric_checks:
    low, high, n = iqr_info(teacher_level_check[col])
    outlier_rows.append({
        'Признак': col,
        'Нижняя граница IQR': round(low, 2),
        'Верхняя граница IQR': round(high, 2),
        'Выбросов': n,
        'Минимум': teacher_level_check[col].min(),
        'Максимум': teacher_level_check[col].max(),
    })

outlier_report = pd.DataFrame(outlier_rows)
outlier_report

,Признак,Нижняя граница IQR,Верхняя граница IQR,Выбросов,Минимум,Максимум
0,age,4.00,84.00,2,16,88
1,experience_age_all,-22.00,66.00,4,0,71
2,experience_age_pedagogic,-26.00,62.00,3,0,67
3,experience_age_other,-6.00,10.00,5881,-33,51
4,n_unique_subjects,-4.00,12.00,223,1,29
5,category_date,"2,016.00","2,024.00",1652,2002,2023
6,category_due,"2,021.00","2,029.00",1679,2000,2100


### 3.1. Возраст и стаж

IQR отмечает несколько очень больших значений возраста и стажа. Они не удаляются, педагог старшего возраста или с большим стажем может быть реальным наблюдением.

Вместо этого проверяем невозможные соотношения:

- отрицательный стаж вне педагогической деятельности невозможен;
- общий или педагогический стаж не может быть больше максимально возможного трудового стажа при данном возрасте. Чтобы не удалить лишние наблюдения считаем, что работа не могла начаться раньше 14 лет.

Для ошибочных сочетаний выполняем корректировку верхней границей Возраст - 14. Строки не удаляются, чтобы не потерять педагога.

Если после ограничения по возрасту педагогический стаж больше общего, оба очищенных
значения и их разность остаются неизвестными (NaN). Исходные значения сохраняются,
`experience_valid=False` отмечает противоречие. Строки не удаляем и стаж нулём не подменяем.

In [71]:
max_possible_exp = (df_clean['age'] - 14).clip(lower=0)

bad_total_exp = df_clean['experience_age_all'] > max_possible_exp
bad_ped_exp = df_clean['experience_age_pedagogic'] > max_possible_exp
negative_other_exp = df_clean['experience_age_other'] < 0

print('Общий стаж больше физически возможного:', int(bad_total_exp.sum()))
print('Педагогический стаж больше физически возможного:', int(bad_ped_exp.sum()))
print('Отрицательный прочий стаж:', int(negative_other_exp.sum()))

# Сохраняем исходные признаки, а для анализа создаём очищенные версии.
df_clean['experience_age_all_clean'] = df_clean['experience_age_all'].clip(upper=max_possible_exp)
df_clean['experience_age_pedagogic_clean'] = df_clean['experience_age_pedagogic'].clip(upper=max_possible_exp)

# Если два стажа противоречат друг другу, не выбираем один из них как верный.
df_clean['experience_valid'] = (
    df_clean['experience_age_pedagogic_clean'].ge(0)
    & df_clean['experience_age_all_clean'].ge(df_clean['experience_age_pedagogic_clean'])
)
df_clean.loc[~df_clean['experience_valid'],
             ['experience_age_all_clean', 'experience_age_pedagogic_clean']] = np.nan
df_clean['experience_age_other_clean'] = (
    df_clean['experience_age_all_clean'] - df_clean['experience_age_pedagogic_clean']
)
print('Записей с неизвестным стажем:',
      df_clean.loc[~df_clean['experience_valid'], ['id', 'year']].drop_duplicates().shape[0])

Общий стаж больше физически возможного: 295
Педагогический стаж больше физически возможного: 289
Отрицательный прочий стаж: 1040
Записей с неизвестным стажем: 119


### 3.2. Даты квалификационной категории

Для дат исправляются только явно невозможные или аномальные значения.

category_date считается некорректной, если категория получена раньше минимально возможного возраста педагога (`год рождения + 14`) или позже года наблюдения. Такие значения заменяются медианой для педагогов того же года и той же квалификационной категории.

Для category_due аномальными считаются слишком далёкие даты окончания действия категории. Они ограничиваются верхней границей, рассчитанной по правилу IQR отдельно для каждого года наблюдения.

Исходные столбцы при этом сохраняются, а обработанные значения записываются в новые признаки.

In [72]:
birth_year = df_clean['year'] - df_clean['age']
invalid_category_date = (
    df_clean['category_date'].notna()
    & ((df_clean['category_date'] < birth_year + 14) | (df_clean['category_date'] > df_clean['year']))
)

print('Невозможных category_date:', int(invalid_category_date.sum()))

df_clean['category_date_clean'] = df_clean['category_date'].astype(float)

# Медианы считаем только по валидным датам.
valid_dates = df_clean.loc[~invalid_category_date].copy()
medians = (
    valid_dates.groupby(['year', 'category_qual_cat'])['category_date']
    .median()
)

for idx in df_clean.index[invalid_category_date]:
    key = (df_clean.at[idx, 'year'], df_clean.at[idx, 'category_qual_cat'])
    replacement = medians.get(key, np.nan)
    df_clean.at[idx, 'category_date_clean'] = replacement

# Для category_due ограничиваем только верхние экстремальные значения по IQR внутри года.
df_clean['category_due_clean'] = df_clean['category_due'].astype(float)
upper_due_by_year = {}
for year, part in df_clean.groupby('year'):
    s = part['category_due'].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    upper_due_by_year[year] = q3 + 1.5 * (q3 - q1)

upper_bounds = df_clean['year'].map(upper_due_by_year)
high_due = df_clean['category_due'].notna() & (df_clean['category_due'] > upper_bounds)
df_clean.loc[high_due, 'category_due_clean'] = upper_bounds[high_due]

print('Аномально большие category_due, скорректированные для анализа:', int(high_due.sum()))
print('Максимум category_due до обработки:', df_clean['category_due'].max())
print('Максимум category_due_clean после обработки:', df_clean['category_due_clean'].max())

Невозможных category_date: 18


Аномально большие category_due, скорректированные для анализа: 235
Максимум category_due до обработки: 2100
Максимум category_due_clean после обработки: 2029.0


### 3.3. Не меняем

Возрастные значения сохраняются, если они физически возможны, даже при выходе за границы IQR. Большое число предметов у одного педагога также не считается ошибкой, поскольку может отражать реальную нагрузку. Старые, но допустимые даты квалификационной категории не удаляются.

Такой подход позволяет исправлять только действительно некорректные значения, не теряя реальные наблюдения.

## 4. Образование, ПК/ПП и Школы: пропуски и выбросы

Таблицы "Образование" и "ПК-ПП" содержат по листу на год (2022–2024), "Школы" - один лист-справочник.
Читаем каждый лист, добавляем столбец с годом и склеиваем.

In [73]:
df_edu = df_edu_all
df_pk = df_pkpp_all

# у "ПК/ПП" очень длинные названия столбцов - сокращаем
df_pk = df_pk.rename(columns={
    'Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК)': 'Тип программы',
    'Дата получения документа/освоения программы': 'Год документа',
})

for name, t in [('Образование', df_edu), ('ПК/ПП', df_pk), ('Школы', df_school)]:
    print(f'{name}: {t.shape[0]:,} строк × {t.shape[1]} столбцов, полных дубликатов: {t.duplicated().sum():,}')

Образование: 302,719 строк × 9 столбцов, полных дубликатов: 68,349
ПК/ПП: 65,733 строк × 7 столбцов, полных дубликатов: 12
Школы: 266 строк × 4 столбцов, полных дубликатов: 0


Полные дубликаты строк не несут новой информации и завышают, например, число пройденных курсов, поэтому удаляем их.

In [74]:
df_edu = df_edu.drop_duplicates().copy()
df_pk = df_pk.drop_duplicates().copy()
print('Образование:', df_edu.shape, '| ПК/ПП:', df_pk.shape)

Образование: (234370, 9) | ПК/ПП: (65721, 7)


### 4.1. Пропуски

Считаем количество и долю пропусков по столбцам. В столбце "Наименование учебного заведения" пропуск часто записан
прочерком `-` - по смыслу это тоже пропуск, поэтому заменяем его на `NaN` до подсчёта.

In [75]:
df_edu.loc[df_edu['edu_org_name'] == '-', 'edu_org_name'] = None


def missing_table(t):
    """Количество и доля пропусков по столбцам (только столбцы с пропусками)."""
    res = pd.DataFrame({'Пропусков': t.isna().sum(), 'Доля, %': (t.isna().mean() * 100).round(2)})
    return res[res['Пропусков'] > 0]


print('ПК/ПП - всего пропусков:', df_pk.isna().sum().sum())
print('Школы - всего пропусков:', df_school.isna().sum().sum())
missing_table(df_edu)

ПК/ПП - всего пропусков: 0
Школы - всего пропусков: 0


,Пропусков,"Доля, %"
edu_org_name,166945,71.23
date_pass,649,0.28
specialization_edu,120683,51.49
qualification,155019,66.14


В "ПК/ПП" и "Школах" пропусков нет. В "Образовании" они есть в четырёх столбцах.
Чтобы выбрать способ заполнения, смотрим долю пропусков по типам образования.

In [76]:
na_cols = ['specialization_edu', 'qualification', 'edu_org_name', 'date_pass']
na_by_type = (df_edu[na_cols].isna().groupby(df_edu['type_education']).mean() * 100).round(1)
na_by_type['Записей'] = df_edu['type_education'].value_counts()
na_by_type.sort_values('Записей', ascending=False)

,specialization_edu,qualification,edu_org_name,date_pass,Записей
type_education,,,,,
Дополнительное профессиональное образование,71.50,92.20,100.00,0.20,166875
Высшее образование,1.30,0.90,0.00,0.50,44747
"Высшее образование - специалитет, магистратура",1.20,0.90,0.10,0.10,8169
Среднее профессиональное образование,1.30,0.70,0.00,0.20,7993
Высшее образование - бакалавриат,1.40,1.90,0.10,0.10,5858
Среднее (полное) общее образование,99.10,98.50,12.20,3.50,343
Аспирантура,15.70,5.60,0.00,0.00,108
Среднее общее образование,91.80,91.80,2.00,0.00,98
Неполное высшее образование,13.60,40.90,0.00,0.00,66


**Вывод и обоснование заполнения**

- Почти все пропуски - у "Дополнительного профессионального образования" (курсы) и общего образования: у них по смыслу
  нет специализации и квалификации диплома, а название заведения не заполняется. У высшего и СПО пропусков 1–2 %.
- **Специализация, Квалификация, Наименование заведения** → `"не указано"`. Это текст: среднее/медиана неприменимы,
  мода исказила бы данные (половина записей получила бы чужую специальность), а удаление строк потеряло бы больше половины таблицы.
- **Дата окончания** → медиана *внутри типа образования*. Пропусков < 1 %, год сильно зависит от типа
  (курсы - недавние, вузы - давние), а медиана устойчива к очень старым годам.

In [77]:
text_cols = ['specialization_edu', 'qualification', 'edu_org_name']
# Флаги сохраняют смысл пропусков после заполнения текста для отображения.
df_edu['has_specialization'] = has_information(df_edu['specialization_edu'])
df_edu['has_qualification'] = has_information(df_edu['qualification'])
df_edu['date_known'] = df_edu['date_pass'].notna()
df_edu[text_cols] = df_edu[text_cols].fillna('не указано')

# Сохраняем прежнее описательное заполнение для отчёта. Выдуманная или исправленная ошибочная дата не становится событием ПК.
median_by_type = df_edu.groupby('type_education')['date_pass'].transform('median')
df_edu['date_pass'] = df_edu['date_pass'].fillna(median_by_type).astype('Int64')
print('Пропусков в «Образовании» осталось:', df_edu.isna().sum().sum())
df_edu.groupby('type_education')['date_pass'].median()

Пропусков в «Образовании» осталось: 0


type_education
Аспирантура                                                  2,019.00
Высшее образование                                           2,001.00
Высшее образование - бакалавриат                             2,019.00
Высшее образование - подготовка кадров высшей квалификации   2,014.00
Высшее образование - специалитет, магистратура               2,007.00
Дополнительное профессиональное образование                  2,018.00
Неполное высшее образование                                  2,020.00
Основное общее образование                                   2,013.00
Среднее (полное) общее образование                           2,011.00
Среднее общее образование                                    2,017.00
Среднее профессиональное образование                         1,997.00
Name: date_pass, dtype: Float64

### 4.2. Выбросы

Используем то же правило IQR, что и для педагогов: выброс - значение вне `[Q1 − 1.5·IQR; Q3 + 1.5·IQR]`.
Проверяем числовые признаки: часы ПК/ПП, год окончания обучения, число зданий и контингент школ
(номер школы - идентификатор, не проверяем).

In [78]:
def detect_outliers_iqr(data, column):
    """Печатает границы IQR и возвращает строки-выбросы."""
    q1, q3 = data[column].quantile([0.25, 0.75])
    lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    print(f'{column}: границы [{lower:.0f}; {upper:.0f}], выбросов {len(outliers):,} '
          f'({len(outliers) / len(data) * 100:.2f}%)')
    return outliers


out_hours = detect_outliers_iqr(df_pk, 'hours_tot')
out_date = detect_outliers_iqr(df_edu, 'date_pass')
out_buildings = detect_outliers_iqr(df_school, 'buildings_num')
out_pupils = detect_outliers_iqr(df_school, 'students_num')

hours_tot: границы [4; 36], выбросов 3,789 (5.77%)
date_pass: границы [1995; 2035], выбросов 20,358 (8.69%)
buildings_num: границы [-8; 28], выбросов 3 (1.13%)
students_num: границы [-1619; 6864], выбросов 1 (0.38%)


#### ПК/ПП - количество часов

In [79]:
px.box(df_pk, x='hours_tot', color='type_prog',
       title='ПК/ПП: количество часов по типу программы').show()
df_pk.groupby('type_prog')['hours_tot'].describe()

,count,mean,std,min,25%,50%,75%,max
type_prog,,,,,,,,
ПК,"65,658.00",23.11,11.36,16.00,16.00,18.00,24.00,182.00
ПП,63.00,260.00,0.00,260.00,260.00,260.00,260.00,260.00


Типичный курс ПК - 16–24 часа, поэтому всё длиннее 36 часов попало в "выбросы". Но это не ошибки:
переподготовка (ПП) длится 260 часов по нормативу, бывают и длинные курсы ПК (72, 182 часа).

**Решение:** исходные часы не меняем - иначе пропадёт вся переподготовка. Для графиков добавляем отдельный
столбец с ограничением по 99-му перцентилю (винзоризация), чтобы шкалу не растягивали редкие длинные программы.

In [80]:
limit = df_pk['hours_tot'].quantile(0.99)
df_pk['Часы (ограничено)'] = df_pk['hours_tot'].clip(upper=limit)
print('Граница (99-й перцентиль):', limit, '| изменено значений:', (df_pk['hours_tot'] > limit).sum())

Граница (99-й перцентиль): 72.0 | изменено значений: 125


#### Образование - год окончания

По IQR выбросами считаются годы раньше ~1995, но у педагогов старше 50 лет диплом 1980-х - норма.
Поэтому проверяем даты по смыслу, сравнивая с возрастом педагога из таблицы педагогов (`df`):
ошибка - окончание раньше 14 лет или год окончания позже года выгрузки.

In [81]:
ages = df[['id', 'year', 'age']].drop_duplicates(['id', 'year'])
df_edu = df_edu.merge(ages, on=['id', 'year'], how='left')

birth_year_edu = df_edu['year'] - df_edu['age']
too_early = (df_edu['date_pass'] - birth_year_edu) < 14
too_late = df_edu['date_pass'] > df_edu['year']

print('Окончили раньше 14 лет:', too_early.sum())
print('Год окончания из будущего:', too_late.sum())
print('Без возраста (нет в таблице педагогов), не проверялись:', df_edu['age'].isna().sum())

Окончили раньше 14 лет: 308
Год окончания из будущего: 8
Без возраста (нет в таблице педагогов), не проверялись: 4004


Это ошибки ввода (например, диплом 1915 года у педагога 1979 г.р.). Само образование у человека есть,
поэтому строки не удаляем, а заменяем год на ближайшее возможное значение (год 14-летия / год выгрузки)
и ставим флаг `Дата исправлена`. Старые, но возможные даты не трогаем.

In [82]:
df_edu['Дата исправлена'] = too_early | too_late
df_edu.loc[too_early, 'date_pass'] = (birth_year_edu[too_early] + 14).astype(int)
df_edu.loc[too_late, 'date_pass'] = df_edu.loc[too_late, 'year']
df_edu = df_edu.drop(columns='age')

print('Исправлено дат:', df_edu['Дата исправлена'].sum())
print('Диапазон лет окончания:', df_edu['date_pass'].min(), '–', df_edu['date_pass'].max())

Исправлено дат: 316
Диапазон лет окончания: 1959 – 2024


#### Школы - здания и контингент

In [83]:
px.box(df_school, x='students_num', hover_data=['school_num', 'district'],
       title='Школы: численность контингента').show()
pd.concat([out_buildings, out_pupils]).drop_duplicates()[['school_num', 'district', 'buildings_num',
                                                           'students_num']]

,school_num,district,buildings_num,students_num
29,46,Инновационный округ,37,6161
36,218,Западный округ,37,4085
88,418,Восточный округ,42,3800
78,201,Южный округ,19,7124


Выбросы - крупные образовательные комплексы, которые реально существуют. "Школы" - справочник,
каждая строка - настоящая школа, поэтому **ничего не удаляем и не меняем**. Для сопоставимости школ разного
размера добавляем показатель "учеников на здание".

In [84]:
df_school['Учеников на здание'] = (df_school['students_num'] / df_school['buildings_num']).round()
df_school['Учеников на здание'].describe()

count     266.00
mean      308.59
std       159.15
min        24.00
25%       214.00
50%       287.00
75%       385.50
max     1,166.00
Name: Учеников на здание, dtype: float64

### 4.3. Как используются очищенные таблицы
Ниже в этом же ноутбуке собирается `teachers.parquet`. Сборка получает `df_clean`, `df_edu`,
`df_pk` и `df_school` из памяти, повторного чтения исходных Excel нет.

Полные повторы дипломов и курсов удалены, разные записи одного человека за год сохраняются.
Смысл отсутствующей специализации и квалификации сохранён флагами до заполнения текста.
Неизвестный год окончания не используется как доказательство недавнего ПК.
Часы ПК/ПП считаются по исходным очищенным часам, столбец с ограничением p99 служит только графикам.

## 5. Кадровая витрина для дашборда и будущей модели
Сохраняем две таблицы: подробную `HW/data/data_clean.parquet` для учебных работ с предметами и параллелями и
`data/teachers.parquet` с одной строкой на `id, year`. Все расчёты находятся в этом ноутбуке.

В кадровой таблице сохраняется контракт дашборда и добавлен прочий стаж для будущей модели.
Количество предметов и параллелей называется `n_subjects` и `n_parallels`.
Кодирование и масштабирование выполняются только внутри будущей обучаемой модели.
Даты категории, описательно скорректированные по IQR/медианам внутри года, не следует
автоматически включать в ML-признаки без отдельной проверки временной методики.

In [85]:
teachers = build_base(df_clean, df_school)
teachers = add_load(teachers, df_clean, df_school)
teachers = add_education(teachers, df_edu)
teachers = add_profdev(teachers, df_edu, df_pk)

assert not teachers.duplicated(['id', 'year']).any()
assert teachers['district'].notna().all()
last = teachers['year'].eq(teachers['year'].max())
assert teachers.loc[last, ['target', 'mobility_status']].isna().all().all()
assert teachers.loc[~last, ['target', 'mobility_status']].notna().all().all()
assert (teachers.loc[~last, 'target'].eq(1) == teachers.loc[~last, 'mobility_status'].eq(LEAVE)).all()
expected = df_clean.drop_duplicates(['id', 'year'])[['id', 'year', 'target']]
check = teachers[['id', 'year', 'target']].merge(expected, on=['id', 'year'], suffixes=('_final', '_detail'))
assert (check['target_final'].eq(check['target_detail']) | check['target_final'].isna() & check['target_detail'].isna()).all()
assert len(teachers) == len(expected)
experience = ['experience_age_all_clean', 'experience_age_pedagogic_clean', 'experience_age_other_clean']
assert teachers.loc[~teachers['experience_valid'], experience].isna().all().all()
assert (teachers.loc[teachers['experience_valid'], 'experience_age_all_clean'] >=
        teachers.loc[teachers['experience_valid'], 'experience_age_pedagogic_clean']).all()
display(teachers.groupby('year').agg(педагогов=('id', 'size'), школ=('school_num', 'nunique'),
    известных_исходов=('target', 'count'), уходов=('target', 'sum')))
display(teachers.groupby('year')[['edu_higher', 'edu_pedagogical', 'pk_recent', 'retraining', 'pk_hours_3y']].mean())
print('Подробная таблица:', df_clean.shape)
print('Кадровая таблица:', teachers.shape)
teachers.head()

,педагогов,школ,известных_исходов,уходов
year,,,,
2022,17396,260,17396,"2,596.00"
2023,17394,265,17394,"2,299.00"
2024,17933,265,0,0.00


,edu_higher,edu_pedagogical,pk_recent,retraining,pk_hours_3y
year,,,,,
2022,0.95,0.80,0.89,0.38,32.69
2023,0.95,0.80,0.93,0.40,60.72
2024,0.96,0.80,0.93,0.40,75.17


Подробная таблица: (417149, 32)
Кадровая таблица: (52723, 36)


,id,year,school_num,district,main_position_cat,age,age_group,experience_age_pedagogic_clean,experience_age_all_clean,category_qual_cat,category_due_clean,class_manager,speciality,sex,academic_degree_cat,experience_age_other_clean,experience_valid,mobility_status,movement_type,target,n_subjects,n_parallels,min_parallel,max_parallel,n_subject_parallel,school_teachers,school_students,school_staff_complete,students_per_teacher,edu_higher,edu_pedagogical,edu_speciality_match,pk_recent,pk_hours_3y,retraining,pk_this_year
0,f583c34a49,2022,478,Восточный округ,учитель,35,35–44,1.00,12.00,не_найдено,NaN,0,не указано,0,не_найдено,11.00,True,0,остался,0.00,2,3,2,9,4,150,3678,1,24.52,1,1,NaN,1,62.00,0,1
1,08ff701e26,2022,523,Западный округ,учитель,54,45–54,24.00,29.00,не_найдено,NaN,0,не указано,0,не_найдено,5.00,True,2,ушёл из системы,1.00,2,5,7,11,6,46,473,1,10.28,1,1,NaN,1,16.00,0,1
2,b813df019b,2022,42,Северный округ,учитель,52,45–54,30.00,34.00,высшая,"2,024.00",1,не указано,0,не_найдено,4.00,True,0,остался,0.00,5,4,5,10,8,109,3103,1,28.47,1,1,NaN,1,64.00,1,1
3,b484e1de85,2022,66,Северный округ,учитель,41,35–44,19.00,19.00,высшая,"2,023.00",1,не указано,0,не_найдено,0.00,True,0,остался,0.00,2,5,3,11,6,80,2040,1,25.50,1,1,NaN,1,60.00,1,1
4,60f45bfe9e,2022,19,Инновационный округ,учитель,58,55–64,21.00,33.00,высшая,"2,023.00",0,не указано,0,не_найдено,12.00,True,0,остался,0.00,2,4,1,6,4,82,1701,1,20.74,1,1,NaN,1,110.00,1,1


## 6. Сохранение двух parquet
Запускайте ноутбук целиком через **Restart and Run All** из `notebooks/` или корня проекта.
Для проверки можно изменить `OUTPUT_DIR` в первой ячейке на временную папку.
По умолчанию `OUTPUT_DIR = ROOT`: последняя ячейка сохраняет
`data/teachers.parquet` и `HW/data/data_clean.parquet` относительно этой папки.
При временном `OUTPUT_DIR` оба результата попадают в такие же подпапки временного каталога. Исходные Excel не меняются.
После обновления `teachers.parquet` перезапустите дашборд: он загружает файл при старте.


In [86]:
for name, table in [('HW/data/data_clean.parquet', df_clean),
                    ('data/teachers.parquet', teachers)]:
    path = OUTPUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_parquet(path, index=False)
    print(path, 'сохранён:', table.shape)


C:\Users\User\PycharmProjects\hr-teachers\tmp\audit-fixes\rebuilt\HW\data\data_clean.parquet сохранён: (417149, 32)
C:\Users\User\PycharmProjects\hr-teachers\tmp\audit-fixes\rebuilt\data\teachers.parquet сохранён: (52723, 36)
